In [ ]:
# Import your uploaded dataset
import os
import zipfile

# Extract your Enhanced REMBG code
with zipfile.ZipFile('/kaggle/input/enhanced-rembg-processor/kaggle_enhanced_rembg_package.zip', 'r') as zip_ref:
    zip_ref.extractall('/kaggle/working')

# Add to Python path
import sys
sys.path.append('/kaggle/working')

print("✅ Enhanced REMBG code extracted and ready!")

In [ ]:
# In notebook cell:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Get R2 credentials from Kaggle secrets
R2_ENDPOINT = f"https://{user_secrets.get_secret('R2_ENDPOINT')}"
R2_ACCESS_KEY = user_secrets.get_secret('R2_ACCESS_KEY')
R2_SECRET_KEY = user_secrets.get_secret('R2_SECRET_KEY')
R2_BUCKET = user_secrets.get_secret('R2_BUCKET')
RENDER_WEBHOOK = user_secrets.get_secret('RENDER_WEBHOOK')

print("✅ R2 credentials loaded from secrets")

In [ ]:
# Install dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
!pip install rembg opencv-python-headless pillow boto3

# Import all necessary modules
import boto3
import zipfile
from pathlib import Path
from datetime import datetime
import time
import requests

# Import your Enhanced REMBG modules
from app.processing.rembg_processor import process_image, initialize_processor
from app.storage.local_storage import LocalStorage
from app.utils.filename_parser import parse_filename

print("🚀 Enhanced REMBG Daily Processor Ready!")

In [ ]:
# Initialize R2 client
r2 = boto3.client('s3',
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY
)

def download_daily_images(job_id):
    """Download today's raw images from R2."""
    print(f"📥 Downloading images for job: {job_id}")

    raw_images = []

    try:
        response = r2.list_objects_v2(
            Bucket=R2_BUCKET,
            Prefix=f'raw_images/{job_id}/'
        )

        for obj in response.get('Contents', []):
            key = obj['Key']
            image_data = r2.get_object(Bucket=R2_BUCKET, Key=key)['Body'].read()

            # Extract symbol number and filename from path
            path_parts = Path(key).parts
            if len(path_parts) >= 3:
                symbol_number = path_parts[2]
                filename = path_parts[3]

                raw_images.append({
                    'bytes': image_data,
                    'filename': filename,
                    'symbol_number': symbol_number
                })

        print(f"✅ Downloaded {len(raw_images)} images")
        return raw_images

    except Exception as e:
        print(f"❌ Error downloading: {e}")
        return []

def process_daily_batch(job_id, image_data_list):
    """Process images with Enhanced REMBG."""
    print(f"🎨 Processing {len(image_data_list)} images...")

    # Initialize Enhanced REMBG
    initialize_processor()

    # Create local storage
    local_storage = LocalStorage(
        upload_dir="/kaggle/working/uploads",
        processed_dir="/kaggle/working/processed"
    )

    processed_files = []
    start_time = time.time()

    for i, image_data in enumerate(image_data_list):
        try:
            # Process with Enhanced REMBG CPU
            processed_buffer = process_image(
                image_data['bytes'],
                output_format="PNG",
                white_background=True,
                optimization_level="cost"
            )

            # Parse filename for symbol number
            parsed = parse_filename(image_data['filename'])
            symbol_number = parsed.symbol_number if parsed.is_valid else image_data.get('symbol_number')

            # Save with same folder structure
            original_name = image_data['filename'].rsplit('.', 1)[0]
            processed_filename = f"{original_name}.png"

            file_path = local_storage.save_processed(
                processed_buffer.read(),
                processed_filename,
                job_id,
                symbol_number=symbol_number
            )

            processed_files.append(file_path)

            # Progress update
            if i % 50 == 0:
                elapsed = time.time() - start_time
                print(f"    📊 Progress: {i+1}/{len(image_data_list)}")

        except Exception as e:
            print(f"    ❌ Failed: {image_data['filename']} - {str(e)}")

    print(f"✅ Processing complete: {len(processed_files)} images")
    return processed_files

def upload_processed_to_r2(job_id, processed_files):
    """Upload processed images back to R2."""
    print(f"📤 Uploading {len(processed_files)} processed images...")

    base_path = Path("/kaggle/working/processed")

    for file_path in processed_files:
        try:
            local_file_path = base_path / file_path

            if local_file_path.exists():
                r2_key = f"processed_images/{file_path}"

                with open(local_file_path, 'rb') as f:
                    r2.put_object(
                        Bucket=R2_BUCKET,
                        Key=r2_key,
                        Body=f.read()
                    )
        except Exception as e:
            print(f"❌ Upload failed: {file_path}")

    print(f"✅ Upload complete")

def create_zip_and_notify(job_id):
    """Create ZIP and notify Render."""
    print("📦 Creating ZIP file...")

    local_storage = LocalStorage(processed_dir="/kaggle/working/processed")

    # Get all processed files
    processed_files = []
    job_dir = Path(f"/kaggle/working/processed/{job_id}")

    if job_dir.exists():
        for file_path in job_dir.rglob("*.png"):
            relative_path = file_path.relative_to(job_dir.parent)
            processed_files.append(str(relative_path))

    if processed_files:
        # Create ZIP with preserved folder structure
        zip_path = local_storage.create_zip(
            processed_files,
            f"processed_{job_id}.zip",
            job_id,
            preserve_structure=True
        )

        # Upload ZIP to R2
        with open(zip_path, 'rb') as f:
            r2.put_object(
                Bucket=R2_BUCKET,
                Key=f"processed_images/{job_id}/processed_{job_id}.zip",
                Body=f.read()
            )

        print(f"✅ ZIP created and uploaded")

    # Notify Render
    try:
        response = requests.post(RENDER_WEBHOOK,
            json={'job_id': job_id, 'status': 'completed'},
            timeout=30)
        print(f"✅ Render notified")
    except:
        print("⚠️ Failed to notify Render")
```

### 4.3 Main Processing Function
```python
def process_daily_job(job_id):
    """Main processing pipeline."""

    print("="*60)
    print(f"🚀 ENHANCED REMBG DAILY PROCESSING")
    print(f"💼 Job ID: {job_id}")
    print(f"💰 Saving $6,900 vs PicWish API")
    print("="*60)

    # Step 1: Download images
    image_data_list = download_daily_images(job_id)

    if not image_data_list:
        print("❌ No images found")
        return

    # Step 2: Process images
    processed_files = process_daily_batch(job_id, image_data_list)

    # Step 3: Upload processed images
    upload_processed_to_r2(job_id, processed_files)

    # Step 4: Create ZIP and notify
    create_zip_and_notify(job_id)

    print("="*60)
    print("🎉 DAILY PROCESSING COMPLETE!")
    print(f"📊 Processed: {len(processed_files)} images")
    print(f"💰 Cost: $0")
    print("="*60)

# Run processing for today
today_job = f"job_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
process_daily_job(today_job)
```

## 🧪 Step 5: Testing Your Setup

### 5.1 Test with Sample Data
```python
# Create test images for validation
import io
from PIL import Image

def create_test_job():
    """Create test images and upload to R2 for testing."""

    test_job_id = f"test_job_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    # Create 3 test images
    test_images = []
    for i in range(3):
        # Create simple test image
        img = Image.new('RGB', (512, 512), color=(100, 150, 200))
        buffer = io.BytesIO()
        img.save(buffer, format='PNG')

        test_images.append({
            'filename': f'58020640_{i+1}_test.png',
            'symbol_number': '58020640',
            'bytes': buffer.getvalue()
        })

    # Upload test images to R2
    for img_data in test_images:
        r2_key = f"raw_images/{test_job_id}/{img_data['symbol_number']}/{img_data['filename']}"

        r2.put_object(
            Bucket=R2_BUCKET,
            Key=r2_key,
            Body=img_data['bytes']
        )

    print(f"✅ Test job created: {test_job_id}")
    return test_job_id

# Run test
test_job = create_test_job()
process_daily_job(test_job)